# HPO dual_view_dupinput_v2 - inherited head configuration

This control does **not** run its own search. It inherits the head configuration
of `dual_view_v2` (`config.HPO_INHERITS`), and here inheriting is not a shortcut
but a requirement: the control *is* the dual-view model, architecture for
architecture and parameter for parameter, differing only in which column feeds
its second branch. Giving it a separate configuration would make the comparison
against `dual_view_v2` two-variable and destroy the one question it exists to
answer - what the morphological information contributes once capacity is held
fixed.

Architectures whose head genuinely differs - `baseline_v2` and `dual_view_v2` -
each run their own exhaustive grid search (`config.HPO_SEARCHES`).

**Output:** `hpo/dual_view_dupinput_v2/best_params.json` (no training happens here)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json
# --- locate the project root -------------------------------------------
# No hardcoded Drive path: take KUSA_ROOT if it is set, otherwise the first
# candidate that actually contains config.py. Works in Colab and locally.
import os, sys
_CANDIDATES = [
    os.environ.get("KUSA_ROOT", ""),
    "/content/drive/MyDrive/kusa",
    "/content/drive/MyDrive/v2_heldout",
    "/content/drive/MyDrive/google_colab/kusa/v2_heldout",
    os.getcwd(),
    os.path.dirname(os.getcwd()),
]
V2_ROOT = next((p for p in _CANDIDATES
                if p and os.path.isfile(os.path.join(p, "config.py"))), None)
assert V2_ROOT, ("config.py not found - set KUSA_ROOT to the project "
                 "directory, e.g. os.environ['KUSA_ROOT'] = '/content/drive/MyDrive/kusa'")
sys.path.insert(0, V2_ROOT)
print("project root:", V2_ROOT)
from config import *
import utils_split as u

In [ ]:
VARIANT = "dual_view_dupinput_v2"
SOURCE  = HPO_INHERITS[VARIANT]          # "dual_view_v2"

SRC_PATH = best_params_path(SOURCE)
assert os.path.exists(SRC_PATH), (
    f"{SRC_PATH} is missing - run hpo_kusa_dual_view_v2 first.")
with open(SRC_PATH, encoding="utf-8") as f:
    src = json.load(f)

print(f"{SOURCE} best params (source of truth):")
for k, v in src.items():
    if not k.startswith("_"):
        print(f"  {k:14s} = {v}")
print(f"  (best HPO-slice value: {src['_best_value']:.4f}, {src['_n_trials']} trials)")

In [ ]:
# The CV notebook reads: batch_size, encoder_lr, head_lr, dropout,
# weight_decay, warmup_ratio, epochs.
best = {
    "batch_size":   src["batch_size"],
    "encoder_lr":   src["encoder_lr"],
    "head_lr":      src["head_lr"],
    "dropout":      src["dropout"],
    "weight_decay": src["weight_decay"],
    "warmup_ratio": src["warmup_ratio"],
    "epochs":       src["epochs"],
    "_variant":          VARIANT,
    "_inherited_from":   SOURCE,
    "_note":             ("identical head architecture; inheriting keeps the "
                          "comparison against the source single-variable"),
    "_best_value":       src["_best_value"],
    "_n_trials":         0,
    "_hpo_slice_seed":   src.get("_hpo_slice_seed"),
    "_hpo_seed":         src.get("_hpo_seed"),
}

out = best_params_path(VARIANT)
with open(out, "w", encoding="utf-8") as f:
    json.dump(best, f, indent=2, ensure_ascii=False)
print(json.dumps(best, indent=2, ensure_ascii=False))
print("\nsaved:", out)

assert_test_untouched(globals())